In [1]:
import tensorflow as tf

# verify GPU presence and make keras run on gpu. Install tf and keras in gpu version 
print("GPU disponibili:", tf.config.list_physical_devices("GPU"))

gpu0 = tf.config.list_physical_devices("GPU")[0]
tf.config.experimental.set_memory_growth(gpu0, True) #helping with VRAM issues



2026-01-15 12:14:06.339977: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/mtebaldi/anaconda3/envs/miki/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


GPU disponibili: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# STEP 1 — load CSVs, merge all except left-out, pad missing features with zeros, make Patient IDs unique (numeric)

import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import List, Tuple

ID_COLS = ["Patient", "Session", "Task"]
LABEL_COL = "FOG"
_FEATURE_PAT = re.compile(r".*_(acc|gyro)_(v|mag)$", re.IGNORECASE)

def build_combined_trainval_and_test(
    dataset_names: List[str],
    left_out: str,
    suffix: str = "_finalmag_inputMOE.csv"
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Loads CSVs one by one and builds:
      - comb_df : concatenation of all datasets except 'left_out'
                  (missing features filled with 0, Patient IDs made numeric)
      - test_df : left-out dataset (kept as-is except padding zeros)
      - feature_cols_union : sorted list of all feature columns
    """

    def detect_feature_columns(df: pd.DataFrame) -> List[str]:
        return sorted([c for c in df.columns if _FEATURE_PAT.match(c)])

    def pad_with_zeros(df: pd.DataFrame, feature_union: List[str]) -> pd.DataFrame:
        df = df.copy()
        for c in feature_union:
            if c not in df.columns:
                df[c] = 0.0
        return df

    # -------- pass 1: scan features (without holding everything)
    feature_union = set()
    for name in dataset_names:
        path = Path(f"{name}{suffix}")
        if not path.exists():
            print(f"⚠️ missing file: {path}")
            continue
        small_df = pd.read_csv(path, nrows=10)  # light read just to detect features
        feature_union.update(detect_feature_columns(small_df))
    feature_union = sorted(feature_union)
    if not feature_union:
        raise ValueError("No feature columns found in any dataset.")

    # -------- pass 2: load + merge on the fly
    comb_parts = []
    test_df = None
    pid_counter = 0
    patient_mapping = {}  # mapping from "Dataset#Patient" -> numeric id

    for name in dataset_names:
        path = Path(f"{name}{suffix}")
        if not path.exists():
            continue
        df = pd.read_csv(path)
        for col in ID_COLS + [LABEL_COL]:
            if col not in df.columns:
                raise ValueError(f"{name} missing required column {col}")
        df["Dataset"] = name
        df = pad_with_zeros(df, feature_union)
        key_series = name + "#" + df["Patient"].astype(str)
        new_ids = []
        for k in key_series:
            if k not in patient_mapping:
                patient_mapping[k] = pid_counter
                pid_counter += 1
            new_ids.append(patient_mapping[k])
        df["Patient_orig"] = df["Patient"]
        df["Patient"] = np.array(new_ids, dtype=np.int32)
        if name == left_out:
            test_df = df  # keep as test
        else:
            
            comb_parts.append(df)

    if test_df is None:
        raise KeyError(f"Left-out dataset '{left_out}' not found among provided names.")

    # merge combined parts
    comb_df = pd.concat(comb_parts, axis=0, ignore_index=True)

    # reorder columns: keep meta first, then features
    meta_cols = ["Dataset"] + ID_COLS + [LABEL_COL]
    features = [c for c in feature_union if c in comb_df.columns]
    others = [c for c in comb_df.columns if c not in meta_cols + features]
    comb_df = comb_df[meta_cols + features + others]
    test_df = test_df[meta_cols + features + [c for c in test_df.columns if c not in meta_cols + features]]

    print(f"✅ Combined: {comb_df.shape}, left-out {left_out}: {test_df.shape}, features: {len(features)}")
    return comb_df, test_df, features


import os
os.chdir('Structured_datasets')
datasets = ['FOGSTAR', 'TDCS', 'IMU', 'Multimodal', 'Oday', 'rempark', 'Daphnet']
DATASET_TO_ID = {ds:i for i, ds in enumerate(datasets)}

comb_df, test_df, feat_union = build_combined_trainval_and_test(
    dataset_names=datasets,
    left_out="FOGSTAR"
)
os.chdir('..')
comb_df 


In [5]:
# STEP 2 — segmentation with majority labeling, respecting (Patient, Session, Task)

import numpy as np
import pandas as pd
from typing import List, Tuple, Optional

def segment_by_registration_majority(
    df: pd.DataFrame,
    feature_cols: List[str],
    *,
    fs: int = 40,
    time:int = 2, 
    overlap: float = 0.8,             # e.g., 0.8 -> 80% overlap
    id_cols: List[str] = ['Dataset', "Patient", "Session", "Task"],
    label_col: str = "FOG",
    majority_threshold: float = 0.5,  # > 0.5 => FOG
    tie_policy: str = "positive",     # if exactly 0.5: 'negative' | 'positive' | 'drop'
    ambiguous_band: Optional[float] = None,  # e.g., 0.1 -> drop frac in [0.4,0.6]
    channel_first: bool = True        # True -> [N, C, T], False -> [N, T, C]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[Tuple]]:
    """
    Segments df into sliding windows without crossing registrations.
    Returns:
      X: np.ndarray  [N, C, T] if channel_first else [N, T, C]
      y: np.ndarray  [N] binary labels
      subjects: np.ndarray [N] (Patient id per window)
      metas: list of tuples (Patient, Session, Task) per window
    """
    # --- checks
    for c in id_cols + [label_col]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in df.")
    for c in feature_cols:
        if c not in df.columns:
            raise ValueError(f"Feature column '{c}' not found in df.")

    window_size = fs*time
    # step from overlap
    if not (0.0 <= overlap < 1.0):
        raise ValueError("overlap must be in [0,1).")
    step = max(1, int(round(window_size * (1.0 - overlap))))

    X_list, y_list, subj_list, dsid_list, meta_list = [], [], [], [], []

    # group by registration
    for key, g in df.groupby(id_cols, sort=False):
        ds,p, s, t = key
        dataset_id = DATASET_TO_ID[ds]
        g = g.reset_index(drop=True)
        if len(g) < window_size:
            continue

        F = g[feature_cols].to_numpy(dtype=float)   # [L, C]
        Y = g[label_col].to_numpy(dtype=float)      # [L]
        L, C = F.shape

        for start in range(0, L - window_size + 1, step):
            end = start + window_size
            win_feat = F[start:end]                 # [T, C]
            win_y = Y[start:end]
            fog_frac = float(win_y.mean())

            # optional ambiguity band (drop windows near 0.5)
            if ambiguous_band is not None and 0.0 < ambiguous_band < 0.5:
                lo, hi = 0.5 - ambiguous_band, 0.5 + ambiguous_band
                if lo <= fog_frac <= hi:
                    continue

            # majority labeling
            if fog_frac > majority_threshold:
                label = 1.0
            elif fog_frac < majority_threshold:
                label = 0.0
            else:  # exact tie
                if tie_policy == "negative":
                    label = 0.0
                elif tie_policy == "positive":
                    label = 1.0
                elif tie_policy == "drop":
                    continue
                else:
                    raise ValueError("tie_policy must be 'negative' | 'positive' | 'drop'.")

            X_list.append(win_feat.T if channel_first else win_feat)
            y_list.append(label)
            dsid_list.append(dataset_id)
            subj_list.append(p)
            meta_list.append((p, s, t))

    if not X_list:
        raise ValueError("No windows produced. Adjust window_size/overlap or remove ambiguity band.")

    X = np.stack(X_list, axis=0)                  # [N,C,T] or [N,T,C]
    y = np.asarray(y_list, dtype=np.float32)      # [N]
    dsid = np.asarray(dsid_list, dtype=np.int32)
    subjects = np.asarray(subj_list)

    return X, y, subjects, dsid, meta_list


# Using comb_df, test_df, feat_union from Step 1
X_comb, y_comb, subj_comb, dsid_comb, meta_comb = segment_by_registration_majority(
    comb_df, feat_union,
    overlap=0.5,
    majority_threshold=0.5,
    tie_policy="positive",     # ties -> FOG
    ambiguous_band=None,       # or 0.1 to drop near-tie windows
    channel_first=True
)

print("Combined windows:", X_comb.shape, "| Pos:", y_comb.shape)
X_comb

In [7]:
# ACTIVITY-BASED FEATURES FOR THE GATING NETWORK
#
# For each expert (sensor-modality pair) and for each window, we compute 
# simple time-domain descriptors of the signal:
#     - mean absolute value
#     - standard deviation
#
# These measures summarize the intensity and variability of the sensor 
# activity over the window. They give the gating network additional context 
# for estimating how "active" or "relevant" each expert is for a specific 
# window, helping the MoE avoid giving weight to sensors that are inactive 
# or uninformative at that moment.

def compute_activity_stats(X, eps=1e-8): 
    """
    X: [N,E,T,2]
    Returns act: [N,E,2] -> (mean_abs, std)
    """
    # collapse channels and time
    # absolute magnitude across v/mag
    abs_sig = np.abs(X)                      # [N,E,T,2]
    abs_sig = abs_sig.reshape(X.shape[0], X.shape[1], -1)  # [N,E,T*2]

    mean_abs = abs_sig.mean(axis=2)           # [N,E]
    std = abs_sig.std(axis=2)                 # [N,E]

    act = np.stack([mean_abs, std], axis=-1)  # [N,E,2]
    return act.astype(np.float32)

#act_comb = compute_activity_stats(X_comb)
#act_comb

array([[[   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        ...,
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ]],

       [[   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        ...,
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ]],

       [[   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        ...,
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ]],

       ...,

       [[ 259.62198  ,    7.8330297],
        [1010.35944  ,    8.008438 ],
        [   0.       ,    0.       ],
        ...,
        [   0.       ,    0.       ],
        [   0.       ,    0.       ],
        [   0.       ,    0.       ]],

       [[ 260.8

In [1]:
# STEP 3 — discover experts, build masks, and compute quantity priors

import re
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple

# pattern for feature columns: "{base}_{acc|gyro}_{v|mag}"
_EXP_PAT = re.compile(r"^(?P<base>.+?)_(?P<mod>(acc|gyro))_(?P<feat>v|mag)$", re.IGNORECASE)

def _parse_expert_cols(columns: List[str]) -> Dict[str, Dict[str, str]]:
    """Return expert -> {'v': col, 'mag': col} from a list of column names."""
    out: Dict[str, Dict[str, str]] = {}
    for c in columns:
        m = _EXP_PAT.match(c)
        if not m:
            continue
        base = m.group("base")
        mod  = m.group("mod").lower()
        feat = m.group("feat").lower()
        key = f"{base}_{mod}".lower()
        out.setdefault(key, {})
        out[key][feat] = c
    return out

def _windows_in_length(L: int, window_size: int, step: int) -> int:
    if L < window_size:
        return 0
    return 1 + (L - window_size) // step

def build_experts_masks_priors(
    dataset_names: List[str],
    left_out: str,
    *,
    window_size: int,
    overlap: float,
    suffix: str = "_finalmag_inputMOE.csv",
    id_cols: List[str] = ["Patient", "Session", "Task"],
    beta: float = 0.5,
) -> Tuple[
    List[str],                    # experts_used (order)
    pd.DataFrame,                 # presence_df (rows=datasets, cols=experts_used)
    Dict[str, Dict[str, Dict]],  # cols_by_dataset[ds][expert] -> {'v': col, 'mag': col}
    Dict[str, np.ndarray],       # mask_by_dataset[ds] -> [E] bool
    np.ndarray,                  # pi  [E]
    np.ndarray,                  # log_pi [E]
    pd.Series                    # windows_per_dataset
]:
    """
    - Inspect CSV headers to discover experts (need BOTH v & mag in a dataset).
    - Keep experts present in at least one TRAIN dataset (any dataset != left_out).
    - Build presence masks per dataset aligned to that expert list.
    - Count windows per dataset using only id columns (low memory).
    - Compute quantity priors: pi_e ∝ (sum_d windows(d) * presence[d,e])^beta.

    Returns:
      experts_used, presence_df, cols_by_dataset, mask_by_dataset, pi, log_pi, windows_per_dataset
    """
    # ----- discover experts and per-dataset columns (header only)
    cols_by_dataset: Dict[str, Dict[str, Dict[str, str]]] = {}
    all_experts = set()

    for ds in dataset_names:
        p = Path(f"Structured_datasets/{ds}{suffix}")
        if not p.exists():
            print(f"⚠️ missing file: {p}")
            continue
        head = pd.read_csv(p, nrows=0)
        parsed = _parse_expert_cols(head.columns.tolist())
        parsed_full = {k: v for k, v in parsed.items() if "v" in v and "mag" in v}
        cols_by_dataset[ds] = parsed_full
        all_experts.update(parsed_full.keys())

    experts_all = sorted(all_experts)

    # presence over all experts
    presence_rows = []
    for ds in dataset_names:
        row = {e: (e in cols_by_dataset.get(ds, {})) for e in experts_all}
        presence_rows.append(pd.Series(row, name=ds))
    presence_all = pd.DataFrame(presence_rows)

    # ----- choose experts used for this split: present in at least one TRAIN dataset
    if left_out not in presence_all.index:
        raise KeyError(f"left_out '{left_out}' not among datasets.")
    train_presence = presence_all.drop(index=left_out)  # training-only
    keep = train_presence.any(axis=0)
    experts_used = keep[keep].index.tolist()

    # (note) If an expert exists in train and also in the left-out dataset, it is naturally kept.
    # If an expert exists ONLY in the left-out dataset (never in train), it is excluded (not trainable).

    # reduce presence to used experts
    presence_df = presence_all[experts_used].copy()

    # build boolean masks aligned to experts_used
    mask_by_dataset = {
        ds: presence_df.loc[ds].to_numpy(dtype=bool)
        for ds in presence_df.index
    }

    # ----- count windows per dataset (low memory)
    step = max(1, int(round(window_size * (1.0 - overlap))))
    win_counts = {}
    for ds in dataset_names:
        p = Path(f"{ds}{suffix}")
        if not p.exists():
            win_counts[ds] = 0
            continue
        # read only id columns to count registration lengths
        usecols = [c for c in id_cols if c]  # ensure list
        df_ids = pd.read_csv(p, usecols=usecols)
        if not all(c in df_ids.columns for c in id_cols):
            raise ValueError(f"{ds} missing id columns {id_cols}")
        g = df_ids.groupby(id_cols, sort=False).size()  # length per registration
        n = 0
        for L in g.to_numpy():
            n += _windows_in_length(int(L), window_size, step)
        win_counts[ds] = int(n)
    windows_per_dataset = pd.Series(win_counts).sort_index()

    # ----- quantity priors over experts (train datasets only)
    train_ds = [d for d in dataset_names if d != left_out]
    pres_train = presence_df.loc[train_ds].to_numpy(dtype=float)   # [D,E]
    win_train = windows_per_dataset.loc[train_ds].to_numpy(dtype=float)  # [D]

    counts_e = (win_train[:, None] * pres_train).sum(axis=0)  # expected #windows seeing expert e
    eps = 1e-8
    weights = np.power(np.maximum(counts_e, eps), beta)
    pi = weights / (weights.sum() + eps)
    log_pi = np.log(np.maximum(pi, eps)).astype(np.float32)

    # filter cols_by_dataset to used experts only
    cols_by_dataset_filtered = {
        ds: {e: cols_by_dataset.get(ds, {}).get(e, {}) for e in experts_used}
        for ds in dataset_names
    }

    presence_df = presence_df[presence_df.sum(axis=0).index]
    experts_used = presence_df.columns.tolist()
    return experts_used, presence_df, cols_by_dataset_filtered, mask_by_dataset, pi.astype(np.float32), log_pi, windows_per_dataset


In [ ]:
datasets = ['FOGSTAR', 'TDCS', 'IMU','Multimodal','Oday', 'rempark', 'Daphnet']
left_out = "FOGSTAR"

os.chdir('Structured_datasets')

fs = 40
time = 2
experts, presence_df, cols_by_ds, mask_by_ds, pi, log_pi, win_counts = build_experts_masks_priors(
    datasets,
    left_out=left_out,
    window_size=fs*time,
    overlap=0.5,
    beta=0.5
)

print("Experts used:", len(experts))
display(presence_df[experts])
print("π :", np.round(pi, 4))
os.chdir('..')

In [10]:
import numpy as np
from typing import List, Tuple

def patient_group_kfold_indices(
    patient_ids: np.ndarray,
    y: np.ndarray,
    *,
    n_splits: int = 5,
    seed: int = 42,
    shuffle: bool = True,
    verbose: bool = True        
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Create patient-wise K folds:
      - split on unique Patient IDs
      - keep all windows of a patient together
      - tries to keep class balance across folds (approx) via patient-level label ratio

    Returns:
      folds: list of (train_idx, val_idx)
    """
    patient_ids = np.asarray(patient_ids)
    y = np.asarray(y).astype(int)
    assert patient_ids.shape[0] == y.shape[0], "patient_ids and y must have same length"

    patients = np.unique(patient_ids)

    # patient-level FOG rate
    p_rate = []
    for p in patients:
        m = (patient_ids == p)
        p_rate.append(y[m].mean() if m.any() else 0.0)
    p_rate = np.asarray(p_rate)

    # stratification bins
    n_bins = min(5, len(patients))
    edges = np.unique(np.quantile(p_rate, np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 3:
        edges = np.unique(np.quantile(p_rate, np.linspace(0, 1, 3)))
    bins = np.digitize(p_rate, edges[1:-1], right=True)

    rng = np.random.default_rng(seed)

    bin_to_patients = {}
    for b in np.unique(bins):
        bin_to_patients[b] = patients[bins == b].copy()
        if shuffle:
            rng.shuffle(bin_to_patients[b])

    # assign patients to folds
    folds_patients = [[] for _ in range(n_splits)]
    for b, plist in bin_to_patients.items():
        for i, p in enumerate(plist):
            folds_patients[i % n_splits].append(p)

    # build indices
    folds = []
    all_idx = np.arange(len(y))

    if verbose:
        print("Fold | #train_win  train_FOG% | #val_win  val_FOG% | #train_pat  #val_pat")
        print("-" * 78)

    for k in range(n_splits):
        val_p = np.array(folds_patients[k])
        val_mask = np.isin(patient_ids, val_p)
        val_idx = all_idx[val_mask]
        train_idx = all_idx[~val_mask]

        folds.append((train_idx, val_idx))

        if verbose:
            tr_y = y[train_idx]
            va_y = y[val_idx]
            tr_pat = np.unique(patient_ids[train_idx])
            va_pat = np.unique(patient_ids[val_idx])

            tr_rate = 100.0 * tr_y.mean() if len(tr_y) else np.nan
            va_rate = 100.0 * va_y.mean() if len(va_y) else np.nan

            print(
                f"{k:>4} | "
                f"{len(train_idx):>9}  {tr_rate:>9.2f} | "
                f"{len(val_idx):>7}  {va_rate:>7.2f} | "
                f"{len(tr_pat):>9}  {len(va_pat):>7}"
            )

    return folds

folds = patient_group_kfold_indices(subj_comb, y_comb, n_splits=5, seed=42)


In [ ]:
import pandas as pd
tmp = pd.DataFrame({"p": subj_comb, "y": y_comb})
stats = tmp.groupby("p")["y"].agg(["count", "mean"]).sort_values("count", ascending=False)
display(stats.head(15))


In [ ]:
import re
import numpy as np

def pack_segmented_X_to_moe(
    X_seg: np.ndarray,          # [N,C,T] channel_first=True
    feature_cols: list,         # length C
    experts: list               # length E, keys like 'wrist_acc'
):
    """
    Returns X_moe [N,E,T,2] where last dim is [v,mag].
    Requires that feature_cols contain '{expert}_v' and '{expert}_mag' (case-insensitive).
    """
    assert X_seg.ndim == 3, f"Expected X_seg [N,C,T], got {X_seg.shape}"
    N, C, T = X_seg.shape
    E = len(experts)

    # map col -> channel index in X_seg
    col_to_idx = {c.lower(): i for i, c in enumerate(feature_cols)}

    X_moe = np.zeros((N, E, T, 2), dtype=np.float32)
    missing = []

    for e_i, ekey in enumerate(experts):
        v_col = f"{ekey}_v".lower()
        m_col = f"{ekey}_mag".lower()

        if v_col in col_to_idx:
            X_moe[:, e_i, :, 0] = X_seg[:, col_to_idx[v_col], :]
        else:
            missing.append(v_col)

        if m_col in col_to_idx:
            X_moe[:, e_i, :, 1] = X_seg[:, col_to_idx[m_col], :]
        else:
            missing.append(m_col)

    if missing:
        print("Warning: missing some expert channels (filled with zeros):", missing[:10], "..." if len(missing)>10 else "")

    return X_moe


def compute_activity_stats_from_moe(X_moe):
    # X_moe: [N,E,T,2]
    abs_sig = np.abs(X_moe).reshape(X_moe.shape[0], X_moe.shape[1], -1)  # [N,E,T*2]
    mean_abs = abs_sig.mean(axis=2)
    std_abs = abs_sig.std(axis=2)
    return np.stack([mean_abs, std_abs], axis=-1).astype(np.float32)     # [N,E,2]
    


In [ ]:
# pack into MoE format
X_comb_moe = pack_segmented_X_to_moe(X_comb, feat_union, experts)  # [N,E,T,2]

# mask per window
presence_mat = presence_df[experts].to_numpy(dtype=bool)  # [D,E]
mask_comb = presence_mat[dsid_comb]                       # [N,E]

# activity stats
act_comb = compute_activity_stats_from_moe(X_comb_moe)     # [N,E,2]

# sanity checks
print("X_comb_moe:", X_comb_moe.shape)
print("mask_comb:", mask_comb.shape)
print("act_comb:", act_comb.shape)
print("log_pi:", log_pi.shape, "E:", len(experts))
assert X_comb_moe.shape[1] == len(experts)
assert mask_comb.shape == (X_comb_moe.shape[0], len(experts))
assert act_comb.shape == (X_comb_moe.shape[0], len(experts), 2)


In [2]:
# =========================
# STEP 6 — Full MoE in Keras + Random Search CV + Early stopping + Class weights + Expert inspection
# =========================

import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score

# -------------------------
# Metrics helpers
# -------------------------
def _safe_auroc(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))

def _safe_auprc(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))

def _best_f1_threshold(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return 0.5, np.nan
    p, r, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2*p*r/(p+r+1e-12)
    i = int(np.nanargmax(f1))
    best_thr = float(thr[i-1]) if i > 0 and (i-1) < len(thr) else 0.5
    y_hat = (y_prob > best_thr).astype(int)
    return best_thr, float(f1_score(y_true, y_hat))

# -------------------------
# Class weights (binary)
# -------------------------
def compute_class_weight_binary(y: np.ndarray) -> dict:
    y = np.asarray(y).astype(int)
    n = len(y)
    n1 = int(np.sum(y == 1))
    n0 = int(np.sum(y == 0))
    if n0 == 0 or n1 == 0:
        return {0: 1.0, 1: 1.0}
    w0 = n / (2.0 * n0)
    w1 = n / (2.0 * n1)
    return {0: float(w0), 1: float(w1)}

# -------------------------
# TF Dataset builder
# -------------------------
def make_tf_dataset(X, y, mask, activity, dsid, batch_size=128, shuffle=True, seed=42):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    mask = mask.astype(bool)
    activity = activity.astype(np.float32)
    dsid = dsid.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((X, mask, activity, dsid, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 50000), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# =========================
# Model components
# =========================
@keras.saving.register_keras_serializable(package="moe")
class ExpertCNN(keras.Model):
    """
    Small 1D CNN that maps [B,T,2] -> [B,d_embed].
    """
    def __init__(self, d_embed=128, width=64, dropout=0.1, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.conv1 = keras.layers.Conv1D(width, 7, padding="same", activation="relu")
        self.conv2 = keras.layers.Conv1D(width, 5, padding="same", activation="relu")
        self.conv3 = keras.layers.Conv1D(width, 3, padding="same", activation="relu")
        self.pool = keras.layers.GlobalAveragePooling1D()
        self.drop = keras.layers.Dropout(dropout)
        self.proj = keras.layers.Dense(d_embed, activation="relu")

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool(x)
        x = self.drop(x, training=training)
        x = self.proj(x)
        return x

    # --- Keras 3 serialization support ---
    def get_config(self):
        config = super().get_config()
        config.update(
            {
                'conv1' : self.conv1, 
                'conv2' : self.conv2, 
                'conv3' : self.conv3, 
                'pool' : self.pool, 
                'drop' : self.drop,
                'proj': self.proj            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        # Keras will call this when loading from .keras
        return cls(**config)

@keras.saving.register_keras_serializable(package="moe")
class GateNet(tf.keras.Model):
    def __init__(self, E, d_gate=64, use_dataset_embed=True, n_datasets=None,
                 d_dataset=16, use_activity=True, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.E = E
        self.use_dataset_embed = use_dataset_embed
        self.use_activity = use_activity
        self.d_gate = d_gate
        self.d_dataset = d_dataset

        if self.use_dataset_embed:
            if n_datasets is None:
                raise ValueError("n_datasets must be provided when use_dataset_embed=True")
            self.dataset_emb = tf.keras.layers.Embedding(int(n_datasets), int(d_dataset))
        else:
            self.dataset_emb = None

        self.mlp1 = tf.keras.layers.Dense(d_gate, activation="relu")
        self.mlp2 = tf.keras.layers.Dense(d_gate, activation="relu")
        self.out = tf.keras.layers.Dense(1, activation=None)

    def call(self, mask, log_pi, dataset_id=None, activity=None, training=False):
        B = tf.shape(mask)[0]
        E = self.E

        log_pi = tf.reshape(log_pi, [1, E, 1])
        log_pi = tf.tile(log_pi, [B, 1, 1])
        m = tf.cast(tf.reshape(mask, [B, E, 1]), tf.float32)
        feats = [log_pi, m]

        if self.use_activity:
            if activity is None:
                activity = tf.zeros([B, E, 2], dtype=tf.float32)
            feats.append(tf.cast(activity, tf.float32))

        if self.use_dataset_embed:
            if dataset_id is None:
                dataset_id = tf.zeros([B], dtype=tf.int32)
            dvec = self.dataset_emb(dataset_id)              # [B, d_dataset]
            dvec = tf.reshape(dvec, [B, 1, self.d_dataset])
            dvec = tf.tile(dvec, [1, E, 1])                  # [B, E, d_dataset]
            feats.append(dvec)

        x = tf.concat(feats, axis=-1)                        # [B, E, F]
        x = self.mlp1(x)
        x = self.mlp2(x)
        logits = self.out(x)[:, :, 0]                        # [B, E]

        neg_inf = tf.constant(-1e9, dtype=tf.float32)
        logits = tf.where(mask, logits, neg_inf)

        alpha = tf.nn.softmax(logits, axis=1)
        alpha = tf.where(tf.math.is_finite(alpha), alpha, tf.zeros_like(alpha))
        return alpha, logits

    # --- Keras 3 serialization support ---
    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "E" : self.E,  
                'use_dataset_embed':self.use_dataset_embed,
                'use_activity' :self.use_activity,
                'd_gate':self.d_gate,
                'd_dataset': self.d_dataset 

            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        # Keras will call this when loading from .keras
        return cls(**config)


@keras.saving.register_keras_serializable(package="moe")
class MoEFOG(tf.keras.Model):
    def __init__(
        self, 
        expert_keys, 
        d_embed=128, 
        expert_width=64, 
        expert_dropout=0.1,
        head_dropout=0.1, 
        gate_kwargs=None, 
        n_datasets=None, 
        name=None, 
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.expert_keys   = list(expert_keys)
        self.E             = len(self.expert_keys)

        # OPTIONAL BUT RECOMMENDED: store hyperparams for future get_config
        self.d_embed       = d_embed
        self.expert_width  = expert_width
        self.expert_dropout = expert_dropout
        self.head_dropout   = head_dropout
        self.gate_kwargs    = gate_kwargs or {}
        self.n_datasets     = n_datasets

        self.experts = [
            ExpertCNN(
                d_embed=d_embed, 
                width=expert_width, 
                dropout=expert_dropout, 
                name=f"expert_{k}",
            )
            for k in self.expert_keys
        ]

        self.gate = GateNet(E=self.E, n_datasets=n_datasets, **(gate_kwargs or {}))

        self.head = tf.keras.Sequential([
            tf.keras.layers.Dropout(head_dropout),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dropout(head_dropout),
            tf.keras.layers.Dense(1, activation=None),
        ])

    def get_config(self):
        # This version is for *new* saves; existing files still contain the old fields.
        config = super().get_config()
        config.update(
            {
                "expert_keys": self.expert_keys,
                "d_embed": self.d_embed,
                "expert_width": self.expert_width,
                "expert_dropout": self.expert_dropout,
                "head_dropout": self.head_dropout,
                "gate_kwargs": self.gate_kwargs,
                "n_datasets": self.n_datasets,
            }
        )
        return config

    def call(self, X, mask, log_pi, dataset_id=None, activity=None, training=False, return_details=False):
        """
        X: [B,E,T,2]
        mask: [B,E] bool
        log_pi: [E]
        """
        # compute expert embeddings
        z_list = []
        for e in range(self.E):
            xe = X[:, e, :, :]                        # [B,T,2]
            ze = self.experts[e](xe, training=training)  # [B,d]
            z_list.append(ze)
        Z = tf.stack(z_list, axis=1)                  # [B,E,d]

        # gating weights
        alpha, gate_logits = self.gate(mask, log_pi, dataset_id=dataset_id, activity=activity, training=training)  # [B,E]

        # mixture
        alpha_exp = tf.expand_dims(alpha, axis=-1)     # [B,E,1]
        mix = tf.reduce_sum(alpha_exp * Z, axis=1)     # [B,d]

        logits = self.head(mix, training=training)[:, 0]  # [B]
        if return_details:
            return logits, alpha, gate_logits
        return logits

    @classmethod
    def from_config(cls, config):
        """
        Config from OLD checkpoints will contain keys like
        'E', 'experts', 'gate', 'head'. We don't want to pass
        those into __init__, so we strip them out and, if needed,
        recover hyperparameters from them.
        """
        config = dict(config)  # make a shallow copy

        # --- BACKWARDS COMPAT: recover hparams from old config if needed ---
        if "experts" in config and "d_embed" not in config:
            # Take the first expert as a template
            first_expert_cfg = config["experts"][0]["config"]
            proj_units = first_expert_cfg["proj"]["config"]["units"]
            width      = first_expert_cfg["conv1"]["config"]["filters"]
            dropout    = first_expert_cfg["drop"]["config"]["rate"]

            config.setdefault("d_embed", proj_units)
            config.setdefault("expert_width", width)
            config.setdefault("expert_dropout", dropout)

        if "gate" in config and "gate_kwargs" not in config:
            gate_cfg = config["gate"]["config"]
            gate_kwargs = {
                "use_dataset_embed": gate_cfg["use_dataset_embed"],
                "use_activity": gate_cfg["use_activity"],
                "d_dataset": gate_cfg["d_dataset"],
                "d_gate": gate_cfg["d_gate"],
            }
            config.setdefault("gate_kwargs", gate_kwargs)

        # NOTE: we *cannot* reliably recover n_datasets from the config
        # (it was never saved). If you used use_dataset_embed=True,
        # you’ll need to add that manually, or adapt GateNet to save it.

        # --- DROP FIELDS THAT __init__ DOESN'T EXPECT ---
        for bad_key in ["E", "experts", "gate", "head"]:
            config.pop(bad_key, None)

        # Anything else (name, trainable, dtype, etc.) can go through.
        return cls(**config)

@keras.saving.register_keras_serializable(package="moe")
class ExpertCNNv2(keras.Model):
    """
    Variant of ExpertCNN using the user's CNN structure:

        Conv1D(width,  kernel_size=3, activation='relu', l2(0.001))
        MaxPooling1D(2)
        Dropout(0.25)

        Conv1D(width//2, kernel_size=3, activation='relu', padding='same', l2(0.001))
        MaxPooling1D(2)
        Dropout(0.25)

        Flatten

        Dense(128, activation='relu', l2(0.001))
        Dropout(0.3)

        Dense(d_embed, activation='relu', l2(0.001))  # embedding for MoE

    Hyperparameters (signature) kept the SAME as original ExpertCNN:
        - d_embed
        - width  (used as first conv filters)
        - dropout (still passed, but internal drops are 0.25 / 0.3 as in your proposal)
    """

    def __init__(
        self,
        d_embed=128,
        width=64,
        dropout=0.1,   # kept for compatibility with original ExpertCNN
        name=None,
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)

        # store for get_config / future use
        self.d_embed = d_embed
        self.width = width
        self.dropout = dropout

        l2_reg = 1e-3  # as in your example

        # --- Block 1 ---
        self.conv1 = layers.Conv1D(
            filters=width,
            kernel_size=3,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg),
        )
        self.pool1 = layers.MaxPooling1D(pool_size=2)
        self.drop1 = layers.Dropout(0.25)

        # --- Block 2 ---
        self.conv2 = layers.Conv1D(
            filters=max(1, width // 2),  # e.g. 64 -> 32
            kernel_size=3,
            activation="relu",
            padding="same",
            kernel_regularizer=regularizers.l2(l2_reg),
        )
        self.pool2 = keras.layers.MaxPooling1D(pool_size=2)
        self.drop2 = keras.layers.Dropout(0.25)

        self.flatten = keras.layers.Flatten()

        # Dense 128 (fixed, as in your proposal)
        self.fc1 = keras.layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=keras.regularizers.l2(l2_reg),
        )
        self.drop3 = keras.layers.Dropout(0.3)

        # Final embedding for MoE (size d_embed, like original ExpertCNN.proj)
        self.proj = keras.layers.Dense(
            d_embed,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg),
        )

    def call(self, x, training=None):
        # x: (batch, time, channels) e.g. (None, 80, 2)
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        x = self.conv2(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop3(x, training=training)
        x = self.proj(x)  # (batch, d_embed)

        return x

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "d_embed": self.d_embed,
                "width": self.width,
                "dropout": self.dropout,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

from keras import layers

@keras.saving.register_keras_serializable(package="moe")
class MoEFOGv2(tf.keras.Model):
    """
    Same interface and behaviour as original MoEFOG,
    but uses ExpertCNNv2 instead of ExpertCNN.
    """

    def __init__(
        self,
        expert_keys,
        d_embed=128,
        expert_width=64,
        expert_dropout=0.1,
        head_dropout=0.1,
        gate_kwargs=None,
        n_datasets=None,
        name=None,
        **kwargs,
    ):
        # Forward base kwargs (trainable, dtype, etc) to tf.keras.Model
        super().__init__(name=name, **kwargs)
        self.expert_keys = list(expert_keys)
        self.E = len(self.expert_keys)

        gate_kwargs = gate_kwargs or {}
        self._gate_kwargs = dict(gate_kwargs)  # keep for get_config
        self.n_datasets = n_datasets

        # --- experts: SAME hyperparams as original, new CNN inside ---
        self.experts = [
            ExpertCNNv2(
                d_embed=d_embed,
                width=expert_width,
                dropout=expert_dropout,
                name=f"expert_{k}",
            )
            for k in self.expert_keys
        ]

        # --- gate: DO NOT TOUCH, use original GateNet ---
        self.gate = GateNet(E=self.E, n_datasets=n_datasets, **gate_kwargs)

        # --- head: same as original MoEFOG ---
        self.head = tf.keras.Sequential(
            [
                tf.keras.layers.Dropout(head_dropout),
                tf.keras.layers.Dense(64, activation="relu"),
                tf.keras.layers.Dropout(head_dropout),
                tf.keras.layers.Dense(1, activation=None),
            ]
        )

    def call(
        self,
        X,
        mask,
        log_pi,
        dataset_id=None,
        activity=None,
        training=False,
        return_details=False,
    ):
        """
        X: [B,E,T,2]
        mask: [B,E] bool
        log_pi: [E]
        dataset_id: [B] (or None)
        activity: [B,E,2] (or None)
        """

        # 1) expert embeddings
        z_list = []
        for e in range(self.E):
            xe = X[:, e, :, :]                              # [B,T,2]
            ze = self.experts[e](xe, training=training)      # [B,d_embed]
            z_list.append(ze)
        Z = tf.stack(z_list, axis=1)                        # [B,E,d_embed]

        # 2) gating weights
        alpha, gate_logits = self.gate(
            mask,
            log_pi,
            dataset_id=dataset_id,
            activity=activity,
            training=training,
        )                                                   # alpha: [B,E]

        # 3) mixture of experts
        alpha_exp = tf.expand_dims(alpha, axis=-1)          # [B,E,1]
        mix = tf.reduce_sum(alpha_exp * Z, axis=1)          # [B,d_embed]

        # 4) head → logits [B]
        logits = self.head(mix, training=training)[:, 0]

        if return_details:
            return logits, alpha, gate_logits
        return logits

    # --- cleaner serialization for v2 (no experts/gate/head in config) ---
    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "expert_keys": self.expert_keys,
                "d_embed": self.experts[0].d_embed,
                "expert_width": self.experts[0].width,
                "expert_dropout": self.experts[0].dropout,
                "head_dropout": self.head.layers[0].rate
                if hasattr(self.head.layers[0], "rate")
                else 0.1,
                "gate_kwargs": self._gate_kwargs,
                "n_datasets": self.n_datasets,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

# =========================
# Training: one fold with early stopping
# =========================
def train_one_fold(
    X_tr, y_tr, mask_tr, act_tr, dsid_tr,
    X_va, y_va, mask_va, act_va, dsid_va,
    *,
    expert_keys,
    log_pi,                 # [E]
    hp: dict,
    class_weight_mode: str, # "none" | "balanced"
    max_epochs: int = 30,
    batch_size: int = 128,
    patience: int = 5,
    seed: int = 42,
    verbose: int = 0, 
):
    keras.utils.set_random_seed(seed)
    
    n_datasets = int(np.max(dsid_tr)) + 1
    
    
 
    model = MoEFOG(
        expert_keys=expert_keys,
        d_embed=int(hp["d_embed"]),
        expert_width=int(hp["expert_width"]),
        expert_dropout=float(hp["expert_dropout"]),
        head_dropout=float(hp["head_dropout"]),
        n_datasets=n_datasets,
        gate_kwargs=dict(
            use_dataset_embed=bool(hp["use_dataset_embed"]),
            use_activity=bool(hp["use_activity"]),
            d_dataset=int(hp.get("d_dataset", 16)),
            d_gate=int(hp.get("d_gate", 64)),
        )
    )

    opt = keras.optimizers.Adam(learning_rate=float(hp["lr"]), clipnorm=1.0)
    bce = keras.losses.BinaryCrossentropy(from_logits=True)

    cw = None
    if class_weight_mode == "balanced":
        cw = compute_class_weight_binary(y_tr)

    ds_tr = make_tf_dataset(X_tr, y_tr, mask_tr, act_tr, dsid_tr, batch_size=batch_size, shuffle=True, seed=seed)
    ds_va = make_tf_dataset(X_va, y_va, mask_va, act_va, dsid_va, batch_size=batch_size, shuffle=False)

    log_pi_tf = tf.constant(log_pi, dtype=tf.float32)

    best_score = -np.inf
    best_weights = None
    best_epoch = -1
    wait = 0

    hist = []

    @tf.function
    def train_step(Xb, Mb, Ab, Dsb, yb):
        with tf.GradientTape() as tape:
            logits = model(Xb, Mb, log_pi_tf, dataset_id=Dsb, activity=Ab, training=True)
            loss = bce(yb, logits)
            if cw is not None:
                w = tf.where(tf.equal(yb, 1.0), cw[1], cw[0])
                loss = tf.reduce_sum(loss * w) / (tf.reduce_sum(w) + 1e-8)
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    def eval_set(ds):
        all_logits, all_y = [], []
        for Xb, Mb, Ab, Dsb, yb in ds:
            logits = model(Xb, Mb, log_pi_tf, dataset_id=Dsb, activity=Ab, training=False)
            all_logits.append(logits.numpy())
            all_y.append(yb.numpy())
        logits = np.concatenate(all_logits, axis=0)
        y = np.concatenate(all_y, axis=0)
        prob = tf.sigmoid(tf.convert_to_tensor(logits)).numpy()
        return logits, y, prob

    for epoch in range(1, max_epochs + 1):
        tr_losses = []
        for Xb, Mb, Ab, Dsb, yb in ds_tr:
            tr_losses.append(float(train_step(Xb, Mb, Ab, Dsb, yb).numpy()))
        tr_loss = float(np.mean(tr_losses)) if tr_losses else np.nan

        _, yv, pv = eval_set(ds_va)
        val_auprc = _safe_auprc(yv, pv)
        val_auroc = _safe_auroc(yv, pv)
        thr, val_f1 = _best_f1_threshold(yv, pv)

        rec = dict(epoch=epoch, train_loss=tr_loss, val_auprc=val_auprc, val_auroc=val_auroc, val_f1=val_f1, val_thr=thr)
        hist.append(rec)

        if verbose:
            print(f"  ep{epoch:02d} loss={tr_loss:.4f} val_auprc={val_auprc:.4f} val_auroc={val_auroc:.4f} val_f1={val_f1:.4f}")

        score = val_auprc if not np.isnan(val_auprc) else -np.inf
        if score > best_score + 1e-6:
            best_score = score
            best_epoch = epoch
            best_weights = model.get_weights()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_weights is not None:
        model.set_weights(best_weights)

    # final metrics on val with best weights
    _, yv, pv = eval_set(ds_va)
    thr, vf1 = _best_f1_threshold(yv, pv)
    val_metrics = {
        "val_best_epoch": int(best_epoch),
        "val_auprc": _safe_auprc(yv, pv),
        "val_auroc": _safe_auroc(yv, pv),
        "val_f1": vf1,
        "val_thr": thr,
    }
    return model, pd.DataFrame(hist), val_metrics


# ========================= 
# Making K-fold split permanent in my experiments
# =========================

def get_or_create_cv_folds(
    patient_ids: np.ndarray,
    y: np.ndarray,
    *,
    n_splits: int = 5,
    seed: int = 42,
    out_dir: str = "moe_runs",
    run_name: str = "lodo_run",
    verbose: bool = True,
):
    """
    Create or load patient-wise CV folds.
    - If a fold file exists, reuse the SAME folds.
    - Otherwise, compute folds once, save to disk, and return them.

    Returns:
        folds: list of (train_idx, val_idx)
    """
    os.makedirs(out_dir, exist_ok=True)
    folds_path = os.path.join(out_dir, f"{run_name}_folds.npz")

    if os.path.exists(folds_path):
        if verbose:
            print(f"Loading existing folds from: {folds_path}")
        data = np.load(folds_path, allow_pickle=True)
        n_splits_saved = int(data["n_splits"])
        assert n_splits_saved == n_splits, f"Saved folds have n_splits={n_splits_saved}, requested {n_splits}"

        folds = []
        for k in range(n_splits):
            tr_idx = data[f"train_{k}"]
            va_idx = data[f"val_{k}"]
            folds.append((tr_idx, va_idx))
        return folds

    # Otherwise: create new folds
    if verbose:
        print(f"No folds file found, creating new folds at: {folds_path}")

    folds = patient_group_kfold_indices(
        patient_ids, y,
        n_splits=n_splits,
        seed=seed,
        shuffle=True
    )

    # save them
    save_dict = {"n_splits": np.array(n_splits, dtype=np.int32)}
    for k, (tr_idx, va_idx) in enumerate(folds):
        save_dict[f"train_{k}"] = np.asarray(tr_idx, dtype=np.int64)
        save_dict[f"val_{k}"] = np.asarray(va_idx, dtype=np.int64)

    np.savez(folds_path, **save_dict)
    if verbose:
        print(f"Saved folds to: {folds_path}")

    return folds
# =========================
# Random search over HP with patient-wise CV
# =========================
def random_search_cv_moe(
    X, y, patients, dsid, mask, activity,
    *,
    expert_keys,
    log_pi,
    n_splits: int = 5,
    n_trials: int = 12,
    max_epochs: int = 12,
    patience: int = 4,
    batch_size: int = 128,
    seed: int = 42,
    out_dir: str = "moe_runs",
    run_name: str = "lodo_run",
    search_space: dict = None,
    verbose: int = 1
):
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.default_rng(seed)

    if search_space is None:
        search_space = {
            "d_embed": [64, 128],
            "expert_width": [32,64],
            "expert_dropout": [0.1, 0.2],
            "head_dropout": [0.1, 0.2],
            "lr": [2e-4, 1e-4],
            "use_dataset_embed": [False],
            "use_activity": [True],
            "d_dataset": [16],
            "d_gate": [32,64],
            "class_weight_mode": ["balanced"],
        }

    folds = get_or_create_cv_folds(
        patients, y,
        n_splits=n_splits,
        seed=seed,
        out_dir=out_dir,
        run_name= f"LODO_leftout_{left_out}",
        verbose=(verbose > 0)
    )


    rows = []
    best_score = -np.inf
    best_hp = None

    for t in range(1, n_trials + 1):
        hp = {k: rng.choice(v) for k, v in search_space.items()}
        class_weight_mode = hp.pop("class_weight_mode")

        if verbose:
            print(f"\nTrial {t}/{n_trials} | cw={class_weight_mode} | hp={hp}")

        fold_metrics = []
        t0 = time.time()

        for k, (tr_idx, va_idx) in enumerate(folds):
            model, hist_df, m = train_one_fold(
                X[tr_idx], y[tr_idx], mask[tr_idx], activity[tr_idx], dsid[tr_idx],
                X[va_idx], y[va_idx], mask[va_idx], activity[va_idx], dsid[va_idx],
                expert_keys=expert_keys,
                log_pi=log_pi,
                hp=hp,
                class_weight_mode=class_weight_mode,
                max_epochs=max_epochs,
                batch_size=batch_size,
                patience=patience,
                seed=seed + 1000*t + k,
                verbose=0
            )
            m["fold"] = k
            fold_metrics.append(m)

        auprcs = [fm["val_auprc"] for fm in fold_metrics]
        aurocs = [fm["val_auroc"] for fm in fold_metrics]
        mean_auprc = float(np.nanmean(auprcs))
        mean_auroc = float(np.nanmean(aurocs))

        row = {
            "trial": t,
            "mean_val_auprc": mean_auprc,
            "mean_val_auroc": mean_auroc,
            "elapsed_sec": float(time.time() - t0),
            "class_weight_mode": class_weight_mode,
            **hp,
            "folds_json": json.dumps(fold_metrics),
        }
        rows.append(row)

        score = mean_auprc if not np.isnan(mean_auprc) else -np.inf
        if score > best_score:
            best_score = score
            best_hp = {"class_weight_mode": class_weight_mode, **hp}

        # incremental save
        results_df = pd.DataFrame(rows).sort_values("mean_val_auprc", ascending=False)
        results_df.to_csv(os.path.join(out_dir, f"{run_name}_trial_results.csv"), index=False)

# =========================
# Train final model on all combined data (with patient holdout for early stop) + save
# =========================
def train_final_moe_and_save(
    X, y, patients, dsid, mask, activity,
    *,
    expert_keys,
    log_pi,
    best_hp: dict,
    folds=None,
    n_splits: int = 5,
    cv_seed: int = 42,
    max_epochs: int = 20,
    patience: int = 5,
    batch_size: int = 128,
    out_dir: str = "moe_runs",
    run_name: str = "lodo_run",
):

    os.makedirs(out_dir, exist_ok=True)

    # --- ensure we have the same CV folds ---
    if folds is None:
        folds = get_or_create_cv_folds(
            patients, y,
            n_splits=n_splits,
            seed=cv_seed,
            out_dir=out_dir,
            run_name=run_name,
            verbose=True
        )

    hp = dict(best_hp)
    class_weight_mode = hp.pop("class_weight_mode")

    models = []
    summaries = []
    model_paths = []

    for k, (tr_idx, va_idx) in enumerate(folds):
        print(f"[Final training] Fold {k+1}/{len(folds)} | "
              f"#train={len(tr_idx)} #val={len(va_idx)}")

        model, hist_df, val_metrics = train_one_fold(
            X[tr_idx], y[tr_idx], mask[tr_idx], activity[tr_idx], dsid[tr_idx],
            X[va_idx], y[va_idx], mask[va_idx], activity[va_idx], dsid[va_idx],
            expert_keys=expert_keys,
            log_pi=log_pi,
            hp=hp,
            class_weight_mode=class_weight_mode,
            max_epochs=max_epochs,
            batch_size=batch_size,
            patience=patience,
            seed=42 + k,
            verbose=0
        )

        # -- Save model --
        model_path = os.path.join(out_dir, f"{run_name}_final_model_fold{k}.keras")
        model.save(model_path)



        # -- Save training curve --
        curve_path = os.path.join(out_dir, f"{run_name}_final_train_curve_fold{k}.csv")
        hist_df.to_csv(curve_path, index=False)

        # -- Store summary into a Python dict (not JSON) --
        summary = {
            "fold": int(k),
            "n_train_windows": int(len(tr_idx)),
            "n_val_windows": int(len(va_idx)),
            "val_auroc": float(val_metrics.get("val_auroc", np.nan)),
            "val_auprc": float(val_metrics.get("val_auprc", np.nan)),
            "val_f1": float(val_metrics.get("val_f1", np.nan)),
            "val_mcc": float(val_metrics.get("val_mcc", np.nan)),
            "val_balacc": float(val_metrics.get("val_balacc", np.nan)),
            "val_thr": float(val_metrics.get("val_thr", np.nan)),
            "best_hp": best_hp,
        }

        summaries.append(summary)
        models.append(model)
        model_paths.append(model_path)

    # ---- SAVE ALL SUMMARIES INTO A SINGLE CSV ----
    all_summaries_df = pd.DataFrame(summaries)
    csv_path = os.path.join(out_dir, f"{run_name}_final_train_summaries_ALL.csv")
    all_summaries_df.to_csv(csv_path, index=False)
    print(f"Saved combined fold summaries to: {csv_path}")

# =====================================

def load_best_hparams_from_trials(
    out_dir: str,
    run_name: str
):
    """
    Load best hyperparameters from a trial results CSV .

    Assumptions:
      - First row is the best trial.
      - Hyperparameter columns include names such as:
            d_embed, expert_width, expert_dropout, head_dropout,
            lr, use_dataset_embed, use_activity, d_dataset, d_gate,
            class_weight_mode
      - Metric columns include: val_auroc, val_auprc, val_f1, val_mcc,
        val_balacc, val_loss, fold, time, etc.
    """

    csv_path = os.path.join(out_dir, f"{run_name}_trial_results.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Trial results CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError(f"Trial results CSV is empty: {csv_path}")

    # Take first row → best trial
    row = df.iloc[0]

    # Columns we exclude (metrics, fold info, runtime…)
    non_hp_cols = {
        "trial", "folds_json", "elapsed_sec" ,
        "mean_val_auroc", "mean_val_auprc"
    }

    # Select hyperparameter columns
    hp_cols = [c for c in df.columns if c not in non_hp_cols]

    # Create dict
    best_hp = {c: row[c] for c in hp_cols}

    return best_hp
# =========================
# Expert inspection: gate usage stats and CSV
# =========================
def expert_gate_inspection_to_csv(
    model: MoEFOG,
    X, y, mask, activity, dsid,
    *,
    expert_keys,
    log_pi,
    out_csv_path: str
):
    log_pi_tf = tf.constant(log_pi, dtype=tf.float32)

    # iterate in batches
    ds = make_tf_dataset(X, y, mask, activity, dsid, batch_size=256, shuffle=False)

    alphas = []
    ys = []
    for Xb, Mb, Ab, Dsb, yb in ds:
        _, alpha, _ = model(Xb, Mb, log_pi_tf, dataset_id=Dsb, activity=Ab, training=False, return_details=True)
        alphas.append(alpha.numpy())
        ys.append(yb.numpy())

    A = np.concatenate(alphas, axis=0)  # [N,E]
    Y = np.concatenate(ys, axis=0).astype(int)

    mean_alpha_all = A.mean(axis=0)
    top1 = A.argmax(axis=1)
    top1_freq = np.bincount(top1, minlength=A.shape[1]) / max(1, len(top1))

    # conditional means
    if np.any(Y == 1):
        mean_alpha_fog = A[Y == 1].mean(axis=0)
    else:
        mean_alpha_fog = np.full((A.shape[1],), np.nan)
    if np.any(Y == 0):
        mean_alpha_nofog = A[Y == 0].mean(axis=0)
    else:
        mean_alpha_nofog = np.full((A.shape[1],), np.nan)

    df = pd.DataFrame({
        "expert": list(expert_keys),
        "mean_gate_weight_all": mean_alpha_all,
        "mean_gate_weight_FOG": mean_alpha_fog,
        "mean_gate_weight_noFOG": mean_alpha_nofog,
        "top1_freq": top1_freq
    }).sort_values("mean_gate_weight_all", ascending=False)

    df.to_csv(out_csv_path, index=False)
    return df


2026-01-20 17:44:37.610703: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/mtebaldi/anaconda3/envs/miki/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [9]:
out_dir = "moe_firsttries"
run_name = "LODO_leftout_FOGSTAR"

# 1) hyperparameter tuning (random search)
"""
random_search_cv_moe(X_comb_moe, y_comb, subj_comb, dsid_comb, mask_comb, act_comb,
    expert_keys=experts,
    log_pi=log_pi,
    n_splits=5,
    n_trials=20,
    max_epochs=20,
    patience=5,
    batch_size=128,
    seed=42,
    out_dir=out_dir,
    run_name=run_name,
    verbose=1
)
"""

'\nrandom_search_cv_moe(X_comb_moe, y_comb, subj_comb, dsid_comb, mask_comb, act_comb,\n    expert_keys=experts,\n    log_pi=log_pi,\n    n_splits=5,\n    n_trials=20,\n    max_epochs=20,\n    patience=5,\n    batch_size=128,\n    seed=42,\n    out_dir=out_dir,\n    run_name=run_name,\n    verbose=1\n)\n'

In [10]:
best_hp = load_best_hparams_from_trials(out_dir, run_name = run_name)
print(best_hp)


{'class_weight_mode': 'balanced', 'd_embed': 128, 'expert_width': 64, 'expert_dropout': 0.1, 'head_dropout': 0.1, 'lr': 0.0002, 'use_dataset_embed': False, 'use_activity': True, 'd_dataset': 16, 'd_gate': 32}


In [ ]:
# 2) train final model on all combined data (patient holdout for early stopping)
train_final_moe_and_save(
    X_comb_moe, y_comb, subj_comb, dsid_comb, mask_comb, act_comb,
    expert_keys=experts,
    log_pi=log_pi,
    best_hp=best_hp,
    folds=None,
    n_splits=5,
    cv_seed=42,
    max_epochs=100,
    patience=5,
    batch_size=256,
    out_dir=out_dir,
    run_name=run_name,
)

In [ ]:
# 1) segment the left-out dataset into windows (same as for combined data)

X_test_seg, y_test, subj_test, dsid_test, meta_test = segment_by_registration_majority(
    test_df,
    feature_cols=feat_union,
    fs=40,
    time=2,
    overlap=0.5,
    id_cols=['Dataset',"Patient", "Session", "Task"],
    label_col="FOG",
    majority_threshold=0.5,
    tie_policy="positive",
    ambiguous_band=None,
    channel_first=True
)

# y_test is now 0/1 per window
print("Test windows:", X_test_seg.shape, "FOG rate:", y_test.mean())

# 2) pack segmented channels into MoE format [N,E,T,2]

X_test_moe = pack_segmented_X_to_moe(
    X_test_seg,   # [N,C,T]
    feat_union,
    experts       # same order as training
)
print("X_test_moe:", X_test_moe.shape)

# 3) build mask and activity stats

presence_mat = presence_df[experts].to_numpy(dtype=bool)   # [D,E]
mask_test = presence_mat[dsid_test]                        # [N,E]

act_test = compute_activity_stats(X_test_moe)              # [N,E,2]

print("mask_test:", mask_test.shape, "act_test:", act_test.shape)


In [40]:
import os, glob, json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, balanced_accuracy_score, matthews_corrcoef
)

# adjust names if your classes differ
custom_objects = {
    "MoEFOG": MoEFOG,
    "ExpertCNN": ExpertCNN,
    "GateNet": GateNet,
}

def predict_proba_moe_from_path(
    model_path,
    X, mask, act, dsid,
    log_pi,
    batch_size=128,
):
    """Load a model from .keras, run predictions, then free it."""
    tf.keras.backend.clear_session()
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)

    log_pi_tf = tf.constant(log_pi, dtype=tf.float32)
    ds = make_tf_dataset(
        X, np.zeros(len(X)), mask, act, dsid,
        batch_size=batch_size,
        shuffle=False
    )

    probs = []
    for Xb, Mb, Ab, Dsb, _ in ds:
        logits = model(Xb, Mb, log_pi_tf, dataset_id=Dsb, activity=Ab, training=False)
        pb = tf.sigmoid(logits).numpy()
        probs.append(pb)

    p = np.concatenate(probs, axis=0)

    # free model & graph
    del model
    tf.keras.backend.clear_session()
    return p


In [41]:
def evaluate_all_folds_on_test(
    out_dir: str,
    run_name: str,
    X_test_moe,
    y_test,
    mask_test,
    act_test,
    dsid_test,
    log_pi,
    batch_size: int = 128,
):
    # find all fold models
    pattern = os.path.join(out_dir, f"{run_name}_final_model_fold*.keras")
    model_paths = sorted(glob.glob(pattern))
    if not model_paths:
        raise FileNotFoundError(f"No models found matching: {pattern}")

    # load fold summaries (to get val_thr per fold)
    summaries_path = os.path.join(out_dir, f"{run_name}_final_train_summaries_ALL.csv")
    summ_df = pd.read_csv(summaries_path)

    y_true = y_test.astype(int)
    test_results = []

    for model_path in model_paths:
        # parse fold index from filename
        # e.g. "..._final_model_fold3.keras" -> fold=3
        base = os.path.basename(model_path)
        fold_str = base.split("fold")[-1].split(".keras")[0]
        fold_id = int(fold_str)

        # get threshold from summary
        row = summ_df[summ_df["fold"] == fold_id]
        if not row.empty and "val_thr" in row.columns:
            thr = float(row["val_thr"].iloc[0])
        else:
            thr = 0.5

        print(f"\n[TEST] Evaluating fold {fold_id} model: {base} | thr={thr:.3f}")

        # predict probabilities with this fold model (loaded then freed)
        p = predict_proba_moe_from_path(
            model_path,
            X_test_moe, mask_test, act_test, dsid_test,
            log_pi,
            batch_size=batch_size,
        )

        y_hat = (p > thr).astype(int)

        if len(np.unique(y_true)) == 2:
            auroc = roc_auc_score(y_true, p)
            auprc = average_precision_score(y_true, p)
            f1    = f1_score(y_true, y_hat)
            bal   = balanced_accuracy_score(y_true, y_hat)
            mcc   = matthews_corrcoef(y_true, y_hat)
        else:
            auroc = auprc = f1 = bal = mcc = np.nan

        res = {
            "fold": fold_id,
            "model_path": model_path,
            "thr": thr,
            "test_auroc": auroc,
            "test_auprc": auprc,
            "test_f1": f1,
            "test_balacc": bal,
            "test_mcc": mcc,
            "n_test_windows": int(len(y_true)),
            "test_fog_rate": float(y_true.mean()),
        }
        test_results.append(res)

    test_results_df = pd.DataFrame(test_results).sort_values("fold").reset_index(drop=True)

    csv_path = os.path.join(out_dir, f"{run_name}_TEST_per_fold.csv")
    test_results_df.to_csv(csv_path, index=False)
    print("Saved per-fold test metrics to:", csv_path)

In [ ]:
def moeLODO(datasets : list, 
            leftout: str,
            hp_fixed: dict, 
            CV_RANDOM = 'OFF',
            time = 2,
            fs = 40,
            out_dir = "moe_runs"
           ):
    
    run_name = f"LODO_CNN_leftout_{left_out}"
    
    os.chdir('Structured_datasets')
    print('LODO initiated: {} as test dataset'.format(leftout))
    DATASET_TO_ID = {ds:i for i, ds in enumerate(datasets)}
    
    comb_df, test_df, feat_union = build_combined_trainval_and_test(
        dataset_names=datasets,
        left_out=leftout
    ) 
    os.chdir('..')   

    print('Combined dataset created')

    
    X_comb, y_comb, subj_comb, dsid_comb, meta_comb = segment_by_registration_majority(
        comb_df, feat_union,
        overlap=0.5,
        majority_threshold=0.5,
        tie_policy="positive",     # ties -> FOG
        ambiguous_band=None,       # or 0.1 to drop near-tie windows
        channel_first=True
    )

    print('segmentation done')

    act_comb = compute_activity_stats(X_comb)
    experts, presence_df, cols_by_ds, mask_by_ds, pi, log_pi, win_counts = build_experts_masks_priors(
        datasets,
        left_out=left_out,
        window_size=fs*time,
        overlap=0.5,
        beta=0.5)

    print('activity and experts computed')
    
    folds = patient_group_kfold_indices(subj_comb, y_comb, n_splits=5, seed=42)
    
    print('patients folds created')
    
    X_comb_moe = pack_segmented_X_to_moe(X_comb, feat_union, experts)  # [N,E,T,2]

    presence_mat = presence_df[experts].to_numpy(dtype=bool)  # [D,E]
    mask_comb = presence_mat[dsid_comb]                       # [N,E]
    act_comb = compute_activity_stats_from_moe(X_comb_moe)     # [N,E,2]
    print("X_comb_moe:", X_comb_moe.shape)
    print("mask_comb:", mask_comb.shape)
    print("act_comb:", act_comb.shape)
    print("log_pi:", log_pi.shape, "E:", len(experts))
    assert X_comb_moe.shape[1] == len(experts)
    assert mask_comb.shape == (X_comb_moe.shape[0], len(experts))
    assert act_comb.shape == (X_comb_moe.shape[0], len(experts), 2)
    print('everything prepared for MOE')
    
    if CV_RANDOM == 'OFF': 
        best_hp = hp_fixed
    else: 
        random_search_cv_moe(X_comb_moe, y_comb, subj_comb, dsid_comb, mask_comb, act_comb,
            expert_keys=experts,
            log_pi=log_pi,
            n_splits=5,
            n_trials=20,
            max_epochs=20,
            patience=5,
            batch_size=256,
            seed=42,
            out_dir=out_dir,
            run_name=run_name,
            verbose=1+3)
        best_hp = load_best_hparams_from_trials(out_dir, run_name)
    print('Hyperparameters ready')

    train_final_moe_and_save(
        X_comb_moe, y_comb, subj_comb, dsid_comb, mask_comb, act_comb,
        expert_keys=experts,
        log_pi=log_pi,
        best_hp=best_hp,
        folds=None,
        n_splits=5,
        cv_seed=42,
        max_epochs=100,
        patience=5,
        batch_size=256,
        out_dir=out_dir,
        run_name=run_name
        )
    
    print('model trained')

    X_test_seg, y_test, subj_test, dsid_test, meta_test = segment_by_registration_majority(
        test_df,feat_union,
        overlap=0.5,
        majority_threshold=0.5,
        tie_policy="positive",     # ties -> FOG
        ambiguous_band=None,       # or 0.1 to drop near-tie windows
        channel_first=True
    )
    X_test_moe = pack_segmented_X_to_moe(
        X_test_seg,
        feat_union,
        experts
    )
    presence_mat = presence_df[experts].to_numpy(dtype=bool)   
    mask_test = presence_mat[dsid_test]                        
    act_test = compute_activity_stats(X_test_moe)          
    evaluate_all_folds_on_test(out_dir = out_dir , run_name = run_name, X_test_moe = X_test_moe,y_test=y_test,mask_test=mask_test,act_test=act_test,dsid_test = dsid_test,log_pi = log_pi)



In [ ]:
datasets = ['FOGSTAR', 'TDCS', 'IMU', 'Multimodal', 'Oday', 'rempark', 'Daphnet']
DATASET_TO_ID = {ds:i for i, ds in enumerate(datasets)}

hp = {'class_weight_mode': 'balanced', 'd_embed': 128, 'expert_width': 64, 'expert_dropout': 0.2, 'head_dropout': 0.2, 'lr': 0.0002, 'use_dataset_embed': False, 'use_activity': True, 'd_dataset': 16, 'd_gate': 32}
for d in datasets:
    left_out = d
    moeLODO(datasets, d, hp)